# Cys-DHE：仅输出最终固定权重集成模型结果

本Notebook保留两个分支的训练过程，因为最终集成概率必须由两个分支共同产生，但不再输出或比较任何单模型性能结果。

固定设置：

- CV划分随机种子：1949
- 网络训练随机种子：1254
- CNN+BiLSTM-Attention权重：0.75
- CNN+Transformer权重：0.25
- 分类阈值：0.5
- 五折交叉验证：StratifiedGroupKFold

最终仅报告：

1. 五折OOF固定权重集成结果
2. 独立测试集固定权重集成结果

并保存最终集成预测文件。


## 说明

两个单分支仍会在后台分别训练，用于产生集成所需的预测概率，但Notebook不显示单分支ACC、SN、SP、MCC、AUROC和AUPR，也不生成单分支性能比较表。


## 分类阈值修改说明

本版本已取消 `prior-aware threshold = 10/11 ≈ 0.9091`。

最终分类采用**默认二分类**：

```python
DEFAULT_THRESHOLD = 0.5
pred = (prob_positive >= 0.5).astype(int)
```

对当前两类别 Softmax 输出而言，这与 `argmax` 选择概率较大的类别等价。
AUROC 和 AUPR 仍使用连续预测概率计算，因此不依赖固定分类阈值。

## 固定集成参数

最终概率计算为：

P(ensemble)=0.75×P(CNN+BiLSTM-Attention)+0.25×P(CNN+Transformer)

最终分类阈值固定为0.5。


In [ ]:
from pathlib import Path
import os
import re
import math
import random
import copy
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    confusion_matrix,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)

TRAIN_FASTA = Path("/content/fold_train.fasta")
TEST_FASTA  = Path("/content/independent_set_test.fasta")

assert TRAIN_FASTA.exists(), (
    "没有找到 /content/fold_train.fasta。"
    "请先在 Colab 左侧 Files 面板上传该文件。"
)
assert TEST_FASTA.exists(), (
    "没有找到 /content/independent_set_test.fasta。"
    "请先在 Colab 左侧 Files 面板上传该文件。"
)

SEED = 1949
CV_SPLIT_SEED = 1949
TORCH_TRAIN_SEED = 1254

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"
AA2IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

SEQ_LEN = 41
CENTER = 20
ALPHA = 0.02

BATCH_SIZE = 64
LR = 5e-4
MAX_EPOCHS = 100
PATIENCE = 15
N_FOLDS = 5

CV_THRESHOLD = 0.5

DEFAULT_THRESHOLD = 0.5

FIXED_TRANSFORMER_WEIGHT = 0.25
FIXED_BILSTM_WEIGHT = 0.75

OUT_DIR = Path("/content/DLBWE_fixed_seed1254_T025_B075_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)


In [ ]:
def read_fasta(path):
    records = []
    header = None
    seq_parts = []

    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts).upper()))
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line)

    if header is not None:
        records.append((header, "".join(seq_parts).upper()))

    return records

def label_from_header(header):
    if header.startswith("+"):
        return 1
    if header.startswith("-"):
        return 0
    raise ValueError(
        f"Header 必须以 + 或 - 标记类别，但发现: {header}"
    )

def protein_group_from_header(header):

    h = header
    if h[:1] in {"+", "-"}:
        h = h[1:]
    if h.startswith("sample"):
        h = h[len("sample"):]
    h = re.sub(r"_\d+$", "", h)
    return h

def validate_records(records, name):
    if not records:
        raise ValueError(f"{name} 为空。")

    bad_len = [(h, len(s)) for h, s in records if len(s) != SEQ_LEN]
    if bad_len:
        raise ValueError(
            f"{name} 中存在非 {SEQ_LEN} aa 序列，例如: {bad_len[:3]}"
        )

    bad_aa = []
    valid = set(AA_ORDER)
    for h, s in records:
        unknown = sorted(set(s) - valid)
        if unknown:
            bad_aa.append((h, unknown))

    if bad_aa:
        raise ValueError(
            f"{name} 中包含非标准氨基酸，例如: {bad_aa[:3]}"
        )

train_records = read_fasta(TRAIN_FASTA)
test_records  = read_fasta(TEST_FASTA)

validate_records(train_records, "fold_train.fasta")
validate_records(test_records, "independent_set_test.fasta")

train_headers = [h for h, s in train_records]
train_seqs    = [s for h, s in train_records]
y_train       = np.array([label_from_header(h) for h in train_headers], dtype=np.int64)
groups        = np.array([protein_group_from_header(h) for h in train_headers])

test_headers  = [h for h, s in test_records]
test_seqs     = [s for h, s in test_records]
y_test        = np.array([label_from_header(h) for h in test_headers], dtype=np.int64)

print("Training samples:", len(y_train))
print("Training positive:", int(y_train.sum()))
print("Training negative:", int((y_train == 0).sum()))
print("Unique training proteins/groups:", len(np.unique(groups)))
print()
print("Independent-test samples:", len(y_test))
print("Independent-test positive:", int(y_test.sum()))
print("Independent-test negative:", int((y_test == 0).sum()))

assert len(y_train) == 1204
assert int(y_train.sum()) == 602
assert int((y_train == 0).sum()) == 602
assert len(y_test) == 913
assert int(y_test.sum()) == 83
assert int((y_test == 0).sum()) == 830


In [ ]:
def encode_bwe(sequences, alpha=ALPHA):

    x = np.zeros(
        (len(sequences), SEQ_LEN, len(AA_ORDER)),
        dtype=np.float32
    )

    distance = np.abs(np.arange(SEQ_LEN) - CENTER)
    weights = np.exp(-alpha * distance).astype(np.float32)

    for n, seq in enumerate(sequences):
        for pos, aa in enumerate(seq):
            x[n, pos, AA2IDX[aa]] = weights[pos]

    return x

X_train = encode_bwe(train_seqs)
X_test  = encode_bwe(test_seqs)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Center positional weight:", float(np.exp(-ALPHA * 0)))
print("Edge positional weight  :", float(np.exp(-ALPHA * 20)))


## 模型结构

```text
41-aa sequence, centered at Cys
        │
        ▼
Binary-Weight Encoding
20-d one-hot × exp(-0.02 × distance)
        │
        ▼
Conv1D: 20 → 64, kernel=4
        │
        ▼
Conv1D: 64 → 128, kernel=4
        │
        ▼
MaxPool2d(kernel=2)
(保留原作者结构)
        │
        ▼
BiLSTM: hidden=64 × 2 directions
        │
        ├──────── BiLSTM sequence outputs [B,17,128]
        │
        ▼
hn [2,B,64]
        │
        ▼
BUG FIX
permute(1,0,2) → reshape(B,128)
        │
        ▼
Bahdanau Attention
        │
        ▼
Context [B,128]
        │
        ▼
FNN: 128 → 20 → 2
        │
        ▼
Softmax probability
        │
        ▼
Default binary classification
Softmax argmax ≡ P(positive) ≥ 0.5
        │
        ▼
Positive / Negative
```

### 相对原附件实现的核心修正

原代码直接：

```python
hn = hn.view(batch, 128)
```

BiLSTM 的 `hn` 实际是 `[direction, batch, hidden] = [2,B,64]`，
因此这里改成：

```python
hn = hn.permute(1,0,2).contiguous().reshape(B,128)
```

这样每个样本的 forward/backward hidden state 才会被正确拼接。

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, in_features=128, hidden_units=64, num_task=1):
        super().__init__()
        self.W1 = nn.Linear(in_features, hidden_units)
        self.W2 = nn.Linear(in_features, hidden_units)
        self.V  = nn.Linear(hidden_units, num_task)

    def forward(self, hidden_states, values):

        hidden_with_time_axis = hidden_states.unsqueeze(1)

        score = self.V(
            torch.tanh(
                self.W1(values) +
                self.W2(hidden_with_time_axis)
            )
        )

        attention_weights = torch.softmax(score, dim=1)

        context_vector = torch.matmul(
            values.transpose(1, 2),
            attention_weights
        ).transpose(1, 2)

        return context_vector, attention_weights

class DLBWECysBugFixed(nn.Module):
    def __init__(self):
        super().__init__()

        self.block2 = nn.Sequential(
            nn.Conv1d(
                in_channels=20,
                out_channels=64,
                kernel_size=4,
                stride=1,
                padding=0
            ),
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=4,
                stride=1,
                padding=0
            ),

            nn.MaxPool2d(kernel_size=2)
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=64,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        self.Attention = BahdanauAttention(
            in_features=128,
            hidden_units=64,
            num_task=1
        )

        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 20),
            nn.ReLU(),
            nn.Linear(20, 2)
        )

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.block2(x)
        x = x.permute(0, 2, 1)

        values, (hn, hc) = self.lstm(x)

        hidden = (
            hn.permute(1, 0, 2)
              .contiguous()
              .reshape(x.size(0), 128)
        )

        context_vector, attention_weights = self.Attention(
            hidden,
            values
        )

        context_vector = context_vector.flatten(1)
        logits = self.classifier(context_vector)

        return logits, attention_weights

_tmp_model = DLBWECysBugFixed().to(DEVICE)
with torch.no_grad():
    _tmp = torch.from_numpy(X_train[:3]).float().to(DEVICE)
    _logits, _att = _tmp_model(_tmp)

print("logits shape:", tuple(_logits.shape))
print("attention shape:", tuple(_att.shape))
del _tmp_model, _tmp, _logits, _att
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def get_logits(model_output):
    """
    Make prediction/training code compatible with both:
      - models returning logits
      - models returning (logits, attention/auxiliary_output)
    """
    if isinstance(model_output, (tuple, list)):
        return model_output[0]
    return model_output

def safe_auroc(y_true, prob):
    return roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan

def safe_aupr(y_true, prob):
    return average_precision_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan

def binary_metrics(y_true, prob_positive, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob_positive = np.asarray(prob_positive, dtype=float)
    y_pred = (prob_positive >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    acc = (tp + tn) / (tp + tn + fp + fn)
    sn = tp / (tp + fn) if (tp + fn) else 0.0
    sp = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (
        2 * precision * sn / (precision + sn)
        if (precision + sn) > 0 else 0.0
    )
    mcc = matthews_corrcoef(y_true, y_pred)

    return {
        "ACC": float(acc),
        "SN": float(sn),
        "SP": float(sp),
        "Precision": float(precision),
        "F1": float(f1),
        "MCC": float(mcc),
        "AUROC": float(safe_auroc(y_true, prob_positive)),
        "AUPR": float(safe_aupr(y_true, prob_positive)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "threshold": float(threshold),
    }

def predict_probabilities(model, X, batch_size=BATCH_SIZE):
    model.eval()
    probs = []

    ds = TensorDataset(torch.from_numpy(X).float())
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(DEVICE)
            logits = get_logits(model(xb))
            p = torch.softmax(logits, dim=1)[:, 1]
            probs.append(p.detach().cpu().numpy())

    return np.concatenate(probs)

def evaluate_loss_and_prob(model, X, y, criterion):
    model.eval()
    ds = TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(y).long()
    )
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    total_loss = 0.0
    total_n = 0
    probs = []

    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = get_logits(model(xb))
            loss = criterion(logits, yb)

            total_loss += loss.item() * len(yb)
            total_n += len(yb)

            p = torch.softmax(logits, dim=1)[:, 1]
            probs.append(p.detach().cpu().numpy())

    return total_loss / total_n, np.concatenate(probs)

DEFAULT_THRESHOLD = 0.5
print("Default binary threshold:", DEFAULT_THRESHOLD)


In [ ]:
def train_one_fold(
    fold_id,
    X_tr, y_tr,
    X_va, y_va,
    seed,
):
    seed_everything(seed)

    model = DLBWECysBugFixed().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    train_ds = TensorDataset(
        torch.from_numpy(X_tr).float(),
        torch.from_numpy(y_tr).long()
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        drop_last=False
    )

    best_mcc = -np.inf
    best_epoch = 0
    best_state = None
    best_val_prob = None
    patience_counter = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for xb, yb in train_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = get_logits(model(xb))
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(yb)
            seen += len(yb)

        train_loss = running_loss / seen

        val_loss, val_prob = evaluate_loss_and_prob(
            model, X_va, y_va, criterion
        )
        m = binary_metrics(
            y_va,
            val_prob,
            threshold=CV_THRESHOLD
        )

        row = {
            "fold": fold_id,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **m,
        }
        history.append(row)

        if m["MCC"] > best_mcc + 1e-8:
            best_mcc = m["MCC"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_val_prob = val_prob.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        OUT_DIR / f"fold_{fold_id}_epoch_metrics.csv",
        index=False
    )

    return model, best_epoch, best_val_prob, history_df


In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=CV_SPLIT_SEED
)

oof_prob = np.full(len(y_train), np.nan, dtype=np.float64)
best_epochs = []

for fold_id, (tr_idx, va_idx) in enumerate(
    sgkf.split(X_train, y_train, groups=groups),
    start=1
):
    tr_groups = set(groups[tr_idx])
    va_groups = set(groups[va_idx])
    assert len(tr_groups.intersection(va_groups)) == 0

    model, best_epoch, best_val_prob, _ = train_one_fold(
        fold_id=fold_id,
        X_tr=X_train[tr_idx],
        y_tr=y_train[tr_idx],
        X_va=X_train[va_idx],
        y_va=y_train[va_idx],
        seed=TORCH_TRAIN_SEED
    )

    oof_prob[va_idx] = best_val_prob
    best_epochs.append(best_epoch)

    torch.save(
        {
            "model_state": model.state_dict(),
            "fold": fold_id,
            "best_epoch": best_epoch,
            "alpha": ALPHA,
            "cv_threshold": CV_THRESHOLD,
            "seed": TORCH_TRAIN_SEED,
        },
        OUT_DIR / f"branch1_fold_{fold_id}_best.pt"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert not np.isnan(oof_prob).any()


In [ ]:
FINAL_EPOCHS = max(int(np.median(best_epochs)), 1)


In [ ]:
seed_everything(TORCH_TRAIN_SEED)

final_model = DLBWECysBugFixed().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=LR)

full_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long()
)

generator = torch.Generator()
generator.manual_seed(TORCH_TRAIN_SEED)

full_dl = DataLoader(
    full_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    drop_last=False
)

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()

    for xb, yb in full_dl:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = get_logits(final_model(xb))
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

torch.save(
    {
        "model_state": final_model.state_dict(),
        "model_name": "CNN+BiLSTM-Attention branch for Cys-DHE ensemble",
        "seq_len": SEQ_LEN,
        "alpha": ALPHA,
        "aa_order": AA_ORDER,
        "final_epochs": FINAL_EPOCHS,
        "seed": TORCH_TRAIN_SEED,
    },
    OUT_DIR / "Cys_DHE_branch1_final.pt"
)


<!-- 单模型独立测试结果已移除 -->


<!-- 原论文与单模型比较已移除 -->


## CNN+Transformer分支

该分支仅用于生成最终固定权重集成所需的第二组预测概率，不单独报告其性能指标。


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:,0::2] = torch.sin(pos * div)
        pe[:,1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self,x):
        return x + self.pe[:,:x.size(1)]

class CNNTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(20,64,4),
            nn.ReLU(),
            nn.Conv1d(64,128,4),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.proj = nn.Linear(128,64)

        self.pos = PositionalEncoding(
            d_model=64,
            max_len=64
        )

        layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=1
        )

        self.fc = nn.Sequential(
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,2)
        )

    def forward(self,x):

        x=x.permute(0,2,1)
        x=self.cnn(x)
        x=x.permute(0,2,1)
        x=self.proj(x)
        x=self.pos(x)
        x=self.encoder(x)
        x=x.mean(dim=1)
        return self.fc(x)


In [ ]:
def train_transformer_fold(fold_id, tr_idx, va_idx):

    seed_everything(TORCH_TRAIN_SEED)

    model=CNNTransformer().to(DEVICE)

    optimizer=torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    criterion=nn.CrossEntropyLoss()

    ds=TensorDataset(
        torch.from_numpy(X_train[tr_idx]).float(),
        torch.from_numpy(y_train[tr_idx]).long()
    )

    generator = torch.Generator()
    generator.manual_seed(TORCH_TRAIN_SEED)

    loader=DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    best=-999
    best_epoch=0
    best_state=None
    best_prob=None

    for epoch in range(1,MAX_EPOCHS+1):

        model.train()

        for xb,yb in loader:
            xb=xb.to(DEVICE)
            yb=yb.to(DEVICE)

            optimizer.zero_grad()

            out=model(xb)
            loss=criterion(out,yb)

            loss.backward()
            optimizer.step()

        prob=predict_probabilities(
            model,
            X_train[va_idx],
            batch_size=BATCH_SIZE
        )

        m=binary_metrics(
            y_train[va_idx],
            prob,
            threshold=0.5
        )

        if m["MCC"]>best:
            best=m["MCC"]
            best_epoch=epoch
            best_state=copy.deepcopy(
                model.state_dict()
            )
            best_prob=prob.copy()

    model.load_state_dict(best_state)

    return model,best_epoch,best_prob


In [ ]:
sgkf_transformer = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=CV_SPLIT_SEED
)

oof_transformer = np.zeros(len(y_train), dtype=np.float64)
transformer_best_epochs = []

for fold_id, (tr_idx, va_idx) in enumerate(
    sgkf_transformer.split(X_train, y_train, groups),
    start=1
):
    _, best_epoch_t, prob = train_transformer_fold(
        fold_id,
        tr_idx,
        va_idx
    )

    transformer_best_epochs.append(best_epoch_t)
    oof_transformer[va_idx] = prob


In [ ]:
best_w = FIXED_TRANSFORMER_WEIGHT

ensemble_oof = (
    FIXED_BILSTM_WEIGHT * oof_prob
    + FIXED_TRANSFORMER_WEIGHT * oof_transformer
)

ensemble_oof_metrics = binary_metrics(
    y_train,
    ensemble_oof,
    threshold=CV_THRESHOLD
)

ensemble_oof_result = pd.DataFrame([{
    "Model": "Cys-DHE Fixed-weight Ensemble",
    "ACC": ensemble_oof_metrics["ACC"],
    "SN": ensemble_oof_metrics["SN"],
    "SP": ensemble_oof_metrics["SP"],
    "MCC": ensemble_oof_metrics["MCC"],
    "AUROC": ensemble_oof_metrics["AUROC"],
    "AUPR": ensemble_oof_metrics["AUPR"],
}])

print("===== FINAL 5-FOLD OOF ENSEMBLE RESULT =====")
display(ensemble_oof_result)

ensemble_oof_result.to_csv(
    OUT_DIR / "FINAL_OOF_ensemble_metrics.csv",
    index=False
)

ensemble_oof_pred_df = pd.DataFrame({
    "header": train_headers,
    "protein_group": groups,
    "true_label": y_train,
    "ensemble_prob_positive": ensemble_oof,
    "ensemble_pred_0.5": (ensemble_oof >= CV_THRESHOLD).astype(int)
})

ensemble_oof_pred_df.to_csv(
    OUT_DIR / "FINAL_OOF_ensemble_predictions.csv",
    index=False
)


## 最终独立测试

以下仅输出Cys-DHE固定权重集成模型在独立测试集上的最终结果。


<!-- 训练阶段单模型及原论文比较已移除 -->


In [ ]:
TRANSFORMER_FINAL_EPOCHS = max(
    int(np.median(transformer_best_epochs)),
    1
)

seed_everything(TORCH_TRAIN_SEED)

final_transformer = CNNTransformer().to(DEVICE)
transformer_criterion = nn.CrossEntropyLoss()
transformer_optimizer = torch.optim.Adam(
    final_transformer.parameters(),
    lr=LR
)

transformer_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long()
)

transformer_generator = torch.Generator()
transformer_generator.manual_seed(TORCH_TRAIN_SEED)

transformer_dl = DataLoader(
    transformer_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=transformer_generator,
    drop_last=False
)

for epoch in range(1, TRANSFORMER_FINAL_EPOCHS + 1):
    final_transformer.train()

    for xb, yb in transformer_dl:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        transformer_optimizer.zero_grad(set_to_none=True)
        logits = get_logits(final_transformer(xb))
        loss = transformer_criterion(logits, yb)
        loss.backward()
        transformer_optimizer.step()

torch.save(
    {
        "model_state": final_transformer.state_dict(),
        "model_name": "CNN+Transformer branch for Cys-DHE ensemble",
        "final_epochs": TRANSFORMER_FINAL_EPOCHS,
        "seed": TORCH_TRAIN_SEED,
        "threshold": DEFAULT_THRESHOLD
    },
    OUT_DIR / "Cys_DHE_branch2_final.pt"
)


In [ ]:
test_prob_bilstm = predict_probabilities(
    final_model,
    X_test,
    batch_size=BATCH_SIZE
)

test_prob_transformer = predict_probabilities(
    final_transformer,
    X_test,
    batch_size=BATCH_SIZE
)

test_prob_ensemble = (
    FIXED_BILSTM_WEIGHT * test_prob_bilstm
    + FIXED_TRANSFORMER_WEIGHT * test_prob_transformer
)

test_ensemble_metrics = binary_metrics(
    y_test,
    test_prob_ensemble,
    threshold=DEFAULT_THRESHOLD
)

final_test_result = pd.DataFrame([{
    "Model": "Cys-DHE Fixed-weight Ensemble",
    "ACC": test_ensemble_metrics["ACC"],
    "SN": test_ensemble_metrics["SN"],
    "SP": test_ensemble_metrics["SP"],
    "MCC": test_ensemble_metrics["MCC"],
    "AUROC": test_ensemble_metrics["AUROC"],
    "AUPR": test_ensemble_metrics["AUPR"],
}])

print("===== FINAL INDEPENDENT TEST ENSEMBLE RESULT =====")
display(final_test_result)

final_test_result.to_csv(
    OUT_DIR / "FINAL_TEST_ensemble_metrics.csv",
    index=False
)

final_test_predictions = pd.DataFrame({
    "header": test_headers,
    "sequence": test_seqs,
    "true_label": y_test,
    "ensemble_prob_positive": test_prob_ensemble,
    "ensemble_pred_0.5": (
        test_prob_ensemble >= DEFAULT_THRESHOLD
    ).astype(int)
})

final_test_predictions.to_csv(
    OUT_DIR / "FINAL_TEST_ensemble_predictions.csv",
    index=False
)

print(
    f"Ensemble weights: "
    f"BiLSTM={FIXED_BILSTM_WEIGHT:.2f}, "
    f"Transformer={FIXED_TRANSFORMER_WEIGHT:.2f}"
)


In [ ]:
fixed_parameter_df = pd.DataFrame([{
    "CV_split_seed": CV_SPLIT_SEED,
    "Torch_training_seed": TORCH_TRAIN_SEED,
    "Transformer_weight": FIXED_TRANSFORMER_WEIGHT,
    "BiLSTM_weight": FIXED_BILSTM_WEIGHT,
    "CV_threshold": CV_THRESHOLD,
    "Test_threshold": DEFAULT_THRESHOLD,
    "Batch_size": BATCH_SIZE,
    "Learning_rate": LR,
    "Max_epochs": MAX_EPOCHS,
    "Patience": PATIENCE,
    "N_folds": N_FOLDS
}])

fixed_parameter_df.to_csv(
    OUT_DIR / "fixed_parameters.csv",
    index=False
)


In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/content/Cys_DHE_Ensemble_Only_outputs",
    "zip",
    root_dir=OUT_DIR
)

print("Final output directory:", OUT_DIR)
print("ZIP archive:", archive_path)


## 输出说明

本Notebook最终只展示两类性能结果：

- 五折OOF的Cys-DHE固定权重集成结果
- 独立测试集的Cys-DHE固定权重集成结果

两个单分支仅用于产生集成概率，不单独展示性能结果。
